# FIFA World Cup 2026 — Notebook 03: Poisson Scoreline Model

## About

**Purpose:** Turn two teams' attack/defence strengths into match outcome probabilities (home win / draw / away win).<br>
**Author:** Ganapathy K<br>
**Date:** 2026-06-06<br>
**Notes:** The match engine. Reads the recency-weighted strengths from notebook 02, converts a fixture into expected goals for each side, builds a full scoreline-probability grid, then sums that grid into win/draw/loss probabilities. The grid uses Poisson with the **Dixon–Coles low-score correction** — plain independent Poisson produces too few low-scoring draws (0-0, 1-1) and too many 1-0/0-1 results, and the `rho` term fixes exactly those four cells. No separate home-advantage term (World Cup games are mostly neutral). Key principle baked in: never pick the single most-likely scoreline — keep the whole distribution and aggregate it.<br>
**Description:** This notebook is the teaching/explainer version. The production code lives in `match_engine.py` (imported by the Monte Carlo notebooks/scripts 05, 08, 09, 10, 11) so the goal model has a single source of truth.

### Change Control

| Date       | Version | Author      | Changes                                  |
|------------|---------|-------------|------------------------------------------|
| 2026-06-06 | 1.0     | Ganapathy K | Initial version                          |
| 2026-06-09 | 1.1     | Ganapathy K | Added Dixon–Coles low-score correction   |


## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

### 1.1 Imports

In [2]:
import pandas as pd
import numpy as np
from scipy.stats import poisson
from pathlib import Path

### 1.2 Config

`BASELINE_GOALS_PER_GAME` is the recency-weighted league average from notebook 02 (its `weighted_baseline`, 1.370). The strengths are stored as ratios relative to this number, so we multiply it back in to recover real expected goals. `MAX_GOALS` caps the scoreline grid — 10 goals a side is far beyond any realistic match, so the probability mass we drop above it is negligible. `DIXON_COLES_RHO` is the low-score correction parameter; `rho < 0` lifts the 0-0 and 1-1 cells (real football has more low draws than independent Poisson predicts) and trims 1-0/0-1. `-0.13` is the Dixon & Coles (1997) estimate.

In [ ]:
PROCESSED_DATA_DIR = Path(r"D:/Data Science/Visual Studio Code/fifa_wc_2026_poisson/data/processed")
TEAM_STRENGTHS_PATH = PROCESSED_DATA_DIR / "team_strengths.parquet"

BASELINE_GOALS_PER_GAME = 1.370   # recency-weighted league average from notebook 02
MAX_GOALS = 10                    # scoreline grid runs 0..MAX_GOALS for each side
DIXON_COLES_RHO = -0.13           # low-score correction; rho < 0 boosts 0-0 and 1-1 draws

## 2. Load Team Strengths

Read the recency-weighted strengths table from notebook 02. Index is `team`; columns are `games_played`, `attack_strength`, `defence_strength` (all ratios × the baseline).

In [4]:
team_strengths = pd.read_parquet(TEAM_STRENGTHS_PATH)
print(f"Loaded strengths for {len(team_strengths)} teams")
team_strengths.head()

Loaded strengths for 336 teams


,games_played,attack_strength,defence_strength
team,,,
Abkhazia,33,1.039349,0.918932
Afghanistan,145,0.821227,1.086270
Albania,397,0.859891,0.855886
Alderney,135,0.921759,1.300346
Algeria,616,1.373497,0.694926


## 3. Expected Goals for a Match

For a fixture, each side's expected goals = **its own attack × the opponent's defence × the baseline**. Attack > 1 means scores more than average; defence < 1 means concedes less than average. So a strong attack against a leaky defence multiplies up; a strong attack against a miserly defence is pulled back down. No home-advantage term in phase 0 — World Cup venues are neutral.

In [5]:
def expected_goals(home_team, away_team):
    home = team_strengths.loc[home_team]
    away = team_strengths.loc[away_team]
    expected_home_goals = home["attack_strength"] * away["defence_strength"] * BASELINE_GOALS_PER_GAME
    expected_away_goals = away["attack_strength"] * home["defence_strength"] * BASELINE_GOALS_PER_GAME
    return expected_home_goals, expected_away_goals

In [6]:
expected_home_goals, expected_away_goals = expected_goals("Spain", "Morocco")
print(f"Spain expected goals:   {expected_home_goals:.2f}")
print(f"Morocco expected goals: {expected_away_goals:.2f}")

Spain expected goals:   1.13
Morocco expected goals: 1.22


## 4. Scoreline Probability Grid

Expected goals is just an average (e.g. 1.8). A real match is a whole count: 0, 1, 2, 3... The **Poisson distribution** turns an average into the probability of each exact count. We compute that for both sides, then take the **outer product** to get the probability of every scoreline combination. `grid[i, j]` = P(home scores `i` **and** away scores `j`).

Plain outer-product Poisson treats the two teams' goals as fully **independent**, and empirically that gets the low scores wrong: real football has *more* 0-0 and 1-1 draws and *fewer* 1-0 / 0-1 results than independence predicts. The **Dixon–Coles correction** fixes exactly those four cells by multiplying them by a factor `tau` governed by one parameter `rho`:

| cell | tau |
|------|-----|
| 0-0 | `1 - λ·μ·rho` |
| 0-1 | `1 + λ·rho` |
| 1-0 | `1 + μ·rho` |
| 1-1 | `1 - rho` |
| else | `1` |

where `λ` = expected home goals, `μ` = expected away goals. With `rho < 0` the 0-0 and 1-1 cells grow and the 1-0/0-1 cells shrink. Applying `tau` breaks the grid's total (it no longer sums to 1), so we **renormalise** at the end.

In [ ]:
def dixon_coles_correction(grid, expected_home_goals, expected_away_goals, rho):
    """Multiply the four low-score cells by the Dixon-Coles tau factors."""
    grid[0, 0] *= 1.0 - expected_home_goals * expected_away_goals * rho
    grid[0, 1] *= 1.0 + expected_home_goals * rho
    grid[1, 0] *= 1.0 + expected_away_goals * rho
    grid[1, 1] *= 1.0 - rho
    return grid


def scoreline_grid(expected_home_goals, expected_away_goals, rho=DIXON_COLES_RHO):
    goals = np.arange(0, MAX_GOALS + 1)
    home_goal_probs = poisson.pmf(goals, expected_home_goals)
    away_goal_probs = poisson.pmf(goals, expected_away_goals)
    grid = np.outer(home_goal_probs, away_goal_probs)
    if rho != 0.0:
        grid = dixon_coles_correction(grid, expected_home_goals, expected_away_goals, rho)
        grid = grid / grid.sum()   # tau breaks normalisation; restore it
    return grid

In [8]:
grid = scoreline_grid(expected_home_goals, expected_away_goals)
print(f"Grid shape: {grid.shape}")
print(f"Total probability captured: {grid.sum():.4f}  (rest is >10 goals, negligible)")

# Show the top-left corner (0-3 goals each) as a readable table
corner = pd.DataFrame(
    grid[:4, :4],
    index=[f"Spain {i}" for i in range(4)],
    columns=[f"Morocco {j}" for j in range(4)],
)
corner.round(3)

Grid shape: (11, 11)
Total probability captured: 1.0000  (rest is >10 goals, negligible)


,Morocco 0,Morocco 1,Morocco 2,Morocco 3
Spain 0,0.095,0.116,0.071,0.029
Spain 1,0.108,0.131,0.080,0.033
Spain 2,0.061,0.074,0.045,0.018
Spain 3,0.023,0.028,0.017,0.007


## 5. Match Outcome — Win / Draw / Loss

Collapse the grid into three numbers. Picture the grid with home goals down the rows and away goals across the columns:
- **Below the diagonal** (home > away) → home win
- **On the diagonal** (home == away) → draw
- **Above the diagonal** (home < away) → away win

Summing each region gives the three outcome probabilities.

In [9]:
def match_outcome_probabilities(home_team, away_team):
    expected_home_goals, expected_away_goals = expected_goals(home_team, away_team)
    grid = scoreline_grid(expected_home_goals, expected_away_goals)
    home_win = np.tril(grid, -1).sum()   # rows below the diagonal: home goals > away goals
    draw = np.trace(grid)                # the diagonal: equal goals
    away_win = np.triu(grid, 1).sum()    # rows above the diagonal: away goals > home goals
    return {"home_win": home_win, "draw": draw, "away_win": away_win}

In [10]:
outcome = match_outcome_probabilities("Spain", "Morocco")
for result, probability in outcome.items():
    print(f"{result:10s} {probability:.1%}")
print(f"{'sum':10s} {sum(outcome.values()):.1%}")

home_win   33.8%
draw       28.0%
away_win   38.2%
sum        100.0%


## 6. Worked Examples

The headline lesson. The single **most-likely exact scoreline** is improbable on its own — it might sit at 9–12%. But its team's chance of *winning* is far higher, because a win is spread across many scorelines (1-0, 2-0, 2-1, 3-1...). This is why you never predict "the score will be X"; you keep the full distribution and sum it into outcomes.

In [11]:
def most_likely_scoreline(home_team, away_team):
    expected_home_goals, expected_away_goals = expected_goals(home_team, away_team)
    grid = scoreline_grid(expected_home_goals, expected_away_goals)
    home_goals, away_goals = np.unravel_index(grid.argmax(), grid.shape)
    return home_goals, away_goals, grid[home_goals, away_goals]

for home_team, away_team in [("Spain", "Morocco"), ("Brazil", "Germany"), ("Canada", "Spain")]:
    home_goals, away_goals, scoreline_prob = most_likely_scoreline(home_team, away_team)
    outcome = match_outcome_probabilities(home_team, away_team)
    print(f"{home_team} vs {away_team}")
    print(f"  most-likely score: {home_goals}-{away_goals} at only {scoreline_prob:.1%}")
    print(f"  outcome: {home_team} win {outcome['home_win']:.0%} | "
          f"draw {outcome['draw']:.0%} | {away_team} win {outcome['away_win']:.0%}")
    print()

Spain vs Morocco
  most-likely score: 1-1 at only 13.1%
  outcome: Spain win 34% | draw 28% | Morocco win 38%

Brazil vs Germany
  most-likely score: 1-1 at only 11.2%
  outcome: Brazil win 42% | draw 24% | Germany win 34%

Canada vs Spain
  most-likely score: 1-1 at only 11.6%
  outcome: Canada win 26% | draw 25% | Spain win 50%

